# 1. Environment Setup & Imports
Mempersiapkan dependensi untuk analisis data.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


# 2. Load Inference & Metadata
Membaca hasil inferensi IndoBERT dan metadata video.

In [ ]:
INFERENCE_PATH = "outputs/inference/full_corpus_predictions.parquet"
METADATA_PATH = "data/processed/video_metadata.csv"

df_preds = pd.read_parquet(INFERENCE_PATH)
df_meta = pd.read_csv(METADATA_PATH)

print(f"Total Comments: {len(df_preds):,}")
print(f"Total Videos in Metadata: {len(df_meta)}")

# Merge
df_merged = df_preds.merge(df_meta, on="video_id", how="left")
print(f"Merged Shape: {df_merged.shape}")


# 3. Global Discourse Distribution

In [ ]:
dist = df_merged["predicted_label"].value_counts().to_dict()
print("Distribution:", dist)

plt.figure(figsize=(10, 5))
sns.barplot(x=list(dist.keys()), y=list(dist.values()), palette="viridis")
plt.title("Discourse Act Distribution (IndoBERT Final)")
plt.ylabel("Count")
plt.show()

# Prepare for JSON export
dashboard_stats = {
    "total_comments": len(df_merged),
    "total_videos": df_merged["video_id"].nunique(),
    "discourse_distribution": dist
}


# 4. Discourse vs Likes Correlation

In [ ]:
# Calculate average likes per discourse act
likes_dist = df_merged.groupby("predicted_label")["like_count"].mean().sort_values(ascending=False).to_dict()
print("Average Likes per Discourse Act:", likes_dist)

plt.figure(figsize=(10, 5))
sns.barplot(x=list(likes_dist.keys()), y=list(likes_dist.values()), palette="magma")
plt.title("Average Likes per Discourse Act")
plt.ylabel("Avg Likes")
plt.show()

dashboard_stats["average_likes_by_act"] = likes_dist


# 5. Video-Level Analysis & Top Videos

In [ ]:
# Group by video_id and predicted_label
video_stats = df_merged.groupby(["video_id", "title", "thumbnail_url", "predicted_label"]).size().unstack(fill_value=0)
video_stats["total_comments"] = video_stats.sum(axis=1)

# Reset index to make title and thumbnail accessible easily
video_stats = video_stats.reset_index()

# Find top 5 videos for each discourse act
top_videos_by_act = {}
labels = df_merged["predicted_label"].unique()

for label in labels:
    if label in video_stats.columns:
        # Sort videos by highest count of this label, get top 5
        top = video_stats.sort_values(by=label, ascending=False).head(5)
        top_list = []
        for _, row in top.iterrows():
            top_list.append({
                "video_id": row["video_id"],
                "title": row["title"],
                "thumbnail_url": row["thumbnail_url"],
                "count": int(row[label]),
                "total_comments": int(row["total_comments"])
            })
        top_videos_by_act[label] = top_list

print("Sample Top Videos for 'Disagreement':")
for v in top_videos_by_act.get("Disagreement", []):
    print(f"- {v['title']} ({v['count']} disagreements)")


# 6. Export Aggregated JSON for Frontend Dashboard

In [ ]:
DASHBOARD_DIR = "src/dashboard/frontend/src/assets/data"
os.makedirs(DASHBOARD_DIR, exist_ok=True)

stats_path = os.path.join(DASHBOARD_DIR, "dashboard_stats.json")
with open(stats_path, "w") as f:
    json.dump(dashboard_stats, f, indent=4)
    
videos_path = os.path.join(DASHBOARD_DIR, "video_stats.json")
with open(videos_path, "w") as f:
    json.dump(top_videos_by_act, f, indent=4)

print(f"Exported dashboard data to {DASHBOARD_DIR}!")
